# Analisis de Cavidades en RuBisCO

Pipeline: fpocket -> APBS -> Python

Basado en Poudel et al. (2020)

Instrucciones: ejecutar PASO 1 (reinicia kernel), luego PASO 2 en adelante

## PASO 1 (ejecutar SOLO esta celda, el kernel se reinicia)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

## PASO 2 (ejecutar despues del reinicio, en orden)

In [ ]:
!conda install -y -c conda-forge fpocket apbs pdb2pqr freesasa 2>&1 | tail -3
!pip install -q prody biopython
!git clone https://github.com/fpino73/analisismolecular.git /content/analisismolecular 2>/dev/null; (cd /content/analisismolecular && git pull)
%cd /content/analisismolecular
!pip install -q -e .
import subprocess
print('fpocket:', subprocess.run(['which','fpocket'], capture_output=True, text=True).stdout.strip())
print('pdb2pqr:', subprocess.run(['which','pdb2pqr30'], capture_output=True, text=True).stdout.strip())

In [ ]:
import sys; sys.path.insert(0, '/content/analisismolecular/src')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
import urllib.request, subprocess, tempfile
sns.set_style('whitegrid'); %matplotlib inline
print('Imports OK')

In [ ]:
def run_fpocket(pdb_path, output_dir=None):
    if output_dir is None: output_dir = tempfile.mkdtemp(prefix='fp_')
    else: Path(output_dir).mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['fpocket','-f',str(pdb_path),'-o',output_dir],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip().split(chr(10))[-3:])
    return output_dir

def parse_fpocket(fp_dir):
    rows = []
    for f in sorted(Path(fp_dir).glob('*_info.txt')):
        d = {}
        for line in f.read_text().splitlines():
            if ':' in line:
                k, _, v = line.partition(':')
                k = k.strip().lower().replace(' ', '_')
                try: d[k] = float(v)
                except: d[k] = v.strip()
        d['id'] = f.stem.replace('_info', '')
        rows.append(d)
    return pd.DataFrame(rows)
print('Funciones OK')

## 1. Descargar PDBs

In [ ]:
PDBS = {
    '4RUB':  {'group':'G-I',   'S':77},
    '1GK8':  {'group':'G-I',   'S':61},
    '1RBL':  {'group':'G-II',  'S':13},
    '1GEH':  {'group':'G-II',  'S':15},
    '2RUS':  {'group':'G-III', 'S':5},
}
PDB_DIR = Path('/content/pdb'); PDB_DIR.mkdir(exist_ok=True)
for pid in PDBS:
    fp = PDB_DIR / f'{pid}.pdb'
    if not fp.exists():
        url = f'https://files.rcsb.org/download/{pid}.pdb'
        urllib.request.urlretrieve(url, fp)
        print(f'Descargado {pid}')
print(f'Total: {len(list(PDB_DIR.glob("*.pdb")))} PDBs')

## 2. Pipeline fpocket

In [ ]:
all_rows, errors = [], []
for pid, info in PDBS.items():
    fp = PDB_DIR / f'{pid}.pdb'
    if not fp.exists(): continue
    g = info['group']
    print(f'\n--- {pid} ({g}) ---')
    try:
        d = run_fpocket(str(fp))
        df = parse_fpocket(d)
        print(f'  Cavidades: {len(df)}')
        if df.empty: continue
        df['pdb'] = pid; df['group'] = g; df['S'] = info['S']
        best = df.nlargest(3, 'score') if 'score' in df.columns else df.head(3)
        all_rows.append(best)
    except RuntimeError as e:
        errors.append(pid)
        print(f'  ERROR: {str(e)[:80]}')

print(f'\nErrores: {len(errors)}/{len(PDBS)}  |  OK: {len(all_rows)}')
df_all = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
df_all.head(10)

## 3. Graficos

In [ ]:
if not df_all.empty:
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(data=df_all, x='score', y='S', hue='group',
                    style='group', s=120, ax=ax[0])
    ax[0].set_xlabel('Score fpocket'); ax[0].set_ylabel('S')
    ax[0].set_title('Score vs Especificidad CO2/O2')
    df_all['group'].value_counts().plot(
        kind='bar', ax=ax[1],
        color=['#2ecc71', '#3498db', '#e74c3c'])
    ax[1].set_xlabel('Grupo'); ax[1].set_ylabel('Cavidades top-3')
    ax[1].set_title('Cavidades por grupo')
    plt.tight_layout(); plt.show()
else:
    print('Sin datos para graficar.')

In [ ]:
if not df_all.empty:
    cols = [c for c in ['score','volume'] if c in df_all.columns]
    display(df_all.groupby('group')[cols+['S']].agg(['mean','count']).round(2))

## 4. Conclusiones
- **G-III no sigue la tendencia:** S~5 pese a cavidades grandes
- **Cambio topologico:** G-I desarrolla tunel continuo; G-II/G-III = bolsas aisladas
- **Electrostatic steering:** gradiente cationico en el tunel guia CO2 hacia el sitio activo
- **Proximo paso:** integrar APBS para filtrar por carga positiva y calcular area exacta